# Notebook 02: Model Training

**Runs on:** Google Colab (T4 GPU)

**Runtime:** ~30 minutes on T4

This notebook trains 4 architectures on EuroSAT:
1. Simple CNN (Baseline)
2. ResNet-50 + Squeeze-and-Excitation Attention
3. Vision Transformer (ViT-Small)
4. ResNet-50 + Custom SSAM (Spectral-Spatial Attention)

Saves: model weights (.pth), training histories (.json), learning curve plots

---
## Instructions
1. Upload this notebook to Google Colab
2. Runtime -> Change runtime type -> T4 GPU
3. Run all cells
4. Download `training_results.zip` at the end

In [ ]:
# Install dependencies
!pip install -q torch torchvision timm scikit-learn tqdm

In [ ]:
import os
import time
import copy
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import EuroSAT
from sklearn.model_selection import train_test_split
from collections import Counter
import timm

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Output directories
os.makedirs('results/models', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/metrics', exist_ok=True)

## 1. Load Dataset and Create Splits

In [ ]:
# Download EuroSAT
raw_dataset = EuroSAT(root='./data', download=True)

CLASS_NAMES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway',
    'Industrial', 'Pasture', 'PermanentCrop', 'Residential',
    'River', 'SeaLake'
]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f"Dataset loaded: {len(raw_dataset)} images, {len(CLASS_NAMES)} classes")

In [ ]:
# Recreate the same stratified split (same SEED as notebook 01)
all_indices = list(range(len(raw_dataset)))
all_labels = [raw_dataset[i][1] for i in all_indices]

train_idx, temp_idx, train_labels, temp_labels = train_test_split(
    all_indices, all_labels, test_size=0.30, random_state=SEED, stratify=all_labels
)
val_idx, test_idx, val_labels, test_labels = train_test_split(
    temp_idx, temp_labels, test_size=0.50, random_state=SEED, stratify=temp_labels
)

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

# Class weights for weighted loss
train_label_counts = Counter(train_labels)
total_train = len(train_labels)
class_weights = torch.tensor(
    [total_train / (len(CLASS_NAMES) * train_label_counts[i]) for i in range(len(CLASS_NAMES))],
    dtype=torch.float32
).to(device)

In [ ]:
# Dataset wrapper
class EuroSATSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        if self.transform:
            img = self.transform(img)
        return img, label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomApply([transforms.RandomRotation(degrees=[90, 90])], p=0.5),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1)], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# DataLoaders
BATCH_SIZE = 64
train_dataset = EuroSATSubset(raw_dataset, train_idx, transform=train_transform)
val_dataset = EuroSATSubset(raw_dataset, val_idx, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 2. Model Architectures

In [ ]:
# ============================================================
# Architecture A: Simple CNN (Baseline)
# ============================================================
class SimpleCNN(nn.Module):
    """
    Simple 4-layer CNN baseline.
    No skip connections, no attention mechanisms.
    """
    def __init__(self, num_classes=10, in_channels=3):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [ ]:
# ============================================================
# Squeeze-and-Excitation (SE) Block
# Hu et al. (2018) - "Squeeze-and-Excitation Networks"
#
# Mathematical formulation:
#   z_c = F_sq(u_c) = (1/H*W) * sum(u_c(i,j))        [Global Avg Pool]
#   s = F_ex(z, W) = sigmoid(W2 * ReLU(W1 * z))       [FC bottleneck]
#   x_hat_c = s_c * u_c                                [Channel rescaling]
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


# ============================================================
# Spectral-Spatial Attention Module (SSAM) - Custom Novel Block
#
# Combines channel attention (which feature channels matter)
# with spatial attention (where in the image to focus).
# Designed for satellite imagery where different spectral bands
# carry different physical information.
# ============================================================
class SSAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super(SSAM, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.channel_mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=spatial_kernel, padding=spatial_kernel // 2, bias=False),
            nn.BatchNorm2d(1),
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.channel_mlp(self.avg_pool(x).view(b, c))
        max_out = self.channel_mlp(self.max_pool(x).view(b, c))
        channel_att = torch.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        x = x * channel_att

        avg_spatial = torch.mean(x, dim=1, keepdim=True)
        max_spatial, _ = torch.max(x, dim=1, keepdim=True)
        spatial_input = torch.cat([avg_spatial, max_spatial], dim=1)
        spatial_att = torch.sigmoid(self.spatial_conv(spatial_input))
        x = x * spatial_att
        return x

In [ ]:
# ============================================================
# Architecture B: ResNet-50 + SE Attention
#
# Why ResNet-50:
#   - Skip connections: H(x) = F(x) + x enables gradient flow
#   - Deeper than ResNet-18: captures complex spatial patterns
#   - ImageNet pretrained: transfer learning for small datasets
#
# Why SE:
#   - Channel attention learns which features matter most
#   - Meaningful for satellite data with varied spectral info
# ============================================================
class ResNetSE(nn.Module):
    def __init__(self, num_classes=10, pretrained=True, use_se=True):
        super(ResNetSE, self).__init__()
        self.use_se = use_se
        self.backbone = timm.create_model('resnet50', pretrained=pretrained, num_classes=0)
        if use_se:
            self.se_block = SEBlock(channels=2048, reduction=16)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        features = self.backbone.forward_features(x)
        if self.use_se:
            features = self.se_block(features)
        features = F.adaptive_avg_pool2d(features, 1).flatten(1)
        return self.classifier(features)


# ============================================================
# Architecture C: Vision Transformer (ViT-Small)
#
# Self-Attention: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V
#
# Why ViT-Small (not Base/Large):
#   - 27K images is small for ViT; avoids overfitting
#   - Compares CNN (local inductive bias) vs Transformer (global)
# ============================================================
class ViTSmall(nn.Module):
    def __init__(self, num_classes=10, pretrained=True):
        super(ViTSmall, self).__init__()
        self.backbone = timm.create_model(
            'vit_small_patch16_224', pretrained=pretrained,
            num_classes=0, img_size=64,
        )
        embed_dim = self.backbone.embed_dim
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)


# ============================================================
# ResNet-50 + SSAM (Custom Spectral-Spatial Attention)
# ============================================================
class ResNetSSAM(nn.Module):
    def __init__(self, num_classes=10, pretrained=True):
        super(ResNetSSAM, self).__init__()
        self.backbone = timm.create_model('resnet50', pretrained=pretrained, num_classes=0)
        self.ssam = SSAM(channels=2048, reduction=16, spatial_kernel=7)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        features = self.backbone.forward_features(x)
        features = self.ssam(features)
        features = F.adaptive_avg_pool2d(features, 1).flatten(1)
        return self.classifier(features)

In [ ]:
# Print model parameter counts
models_info = {
    'Simple CNN': SimpleCNN(),
    'ResNet-50+SE': ResNetSE(pretrained=False),
    'ViT-Small': ViTSmall(pretrained=False),
    'ResNet-50+SSAM': ResNetSSAM(pretrained=False),
}

print("Model Parameter Counts:")
print("-" * 45)
for name, model in models_info.items():
    params = sum(p.numel() for p in model.parameters())
    print(f"  {name:<20} {params:>12,} params")
del models_info

## 3. Training Infrastructure

- **Optimizer:** AdamW (decoupled weight decay, better than Adam for regularization)
- **LR Schedule:** Cosine annealing (smooth LR decay)
- **Loss:** CrossEntropy with label smoothing (0.1) + class weights
- **Early Stopping:** Patience=10 on validation loss

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.should_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / total, correct / total


def train_model(model, train_loader, val_loader, model_name,
                num_epochs=50, lr=1e-3, weight_decay=1e-4, patience=10):
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")

    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    early_stopping = EarlyStopping(patience=patience)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
    best_val_acc = 0.0
    best_model_state = None
    start_time = time.time()

    for epoch in range(num_epochs):
        current_lr = optimizer.param_groups[0]['lr']
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = copy.deepcopy(model.state_dict())

        if (epoch + 1) % 5 == 0 or epoch == 0:
            elapsed = time.time() - start_time
            print(f"Epoch {epoch+1:3d}/{num_epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
                  f"LR: {current_lr:.2e} | {elapsed:.0f}s")

        early_stopping(val_loss)
        if early_stopping.should_stop:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_model_state)
    total_time = time.time() - start_time
    print(f"Done in {total_time:.0f}s | Best val acc: {best_val_acc:.4f}")

    # Save model weights
    torch.save(model.state_dict(), f'results/models/{model_name}.pth')
    # Save history
    with open(f'results/metrics/{model_name}_history.json', 'w') as f:
        json.dump(history, f)

    return history, best_val_acc


print("Training infrastructure ready.")

## 4. Train All Models

In [ ]:
all_histories = {}

In [ ]:
# Train Simple CNN
model_cnn = SimpleCNN(num_classes=10).to(device)
h, _ = train_model(model_cnn, train_loader, val_loader, 'SimpleCNN',
                    num_epochs=50, lr=1e-3, patience=10)
all_histories['Simple CNN'] = h
del model_cnn
torch.cuda.empty_cache()

In [ ]:
# Train ResNet-50 + SE
model_resnet = ResNetSE(num_classes=10, pretrained=True, use_se=True).to(device)
h, _ = train_model(model_resnet, train_loader, val_loader, 'ResNet50_SE',
                    num_epochs=30, lr=1e-4, patience=10)
all_histories['ResNet-50+SE'] = h
del model_resnet
torch.cuda.empty_cache()

In [ ]:
# Train ViT-Small
model_vit = ViTSmall(num_classes=10, pretrained=True).to(device)
h, _ = train_model(model_vit, train_loader, val_loader, 'ViT_Small',
                    num_epochs=30, lr=1e-4, patience=10)
all_histories['ViT-Small'] = h
del model_vit
torch.cuda.empty_cache()

In [ ]:
# Train ResNet-50 + SSAM (Custom)
model_ssam = ResNetSSAM(num_classes=10, pretrained=True).to(device)
h, _ = train_model(model_ssam, train_loader, val_loader, 'ResNet50_SSAM',
                    num_epochs=30, lr=1e-4, patience=10)
all_histories['ResNet-50+SSAM'] = h
del model_ssam
torch.cuda.empty_cache()

## 5. Learning Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = {'Simple CNN': '#e74c3c', 'ResNet-50+SE': '#2ecc71',
          'ViT-Small': '#3498db', 'ResNet-50+SSAM': '#9b59b6'}

for name, history in all_histories.items():
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0][0].plot(epochs, history['train_loss'], label=name, color=colors[name], linewidth=2)
    axes[0][1].plot(epochs, history['val_loss'], label=name, color=colors[name], linewidth=2)
    axes[1][0].plot(epochs, [a*100 for a in history['train_acc']], label=name, color=colors[name], linewidth=2)
    axes[1][1].plot(epochs, [a*100 for a in history['val_acc']], label=name, color=colors[name], linewidth=2)

titles = ['Training Loss', 'Validation Loss', 'Training Accuracy (%)', 'Validation Accuracy (%)']
ylabels = ['Loss', 'Loss', 'Accuracy (%)', 'Accuracy (%)']
for ax, title, ylabel in zip(axes.flatten(), titles, ylabels):
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves - All Architectures', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Download Results

In [ ]:
# List saved files
print("Saved files:")
for root, dirs, files in os.walk('results'):
    for f in sorted(files):
        filepath = os.path.join(root, f)
        size = os.path.getsize(filepath) / 1024
        print(f"  {filepath:<45} ({size:.0f} KB)")

# Zip for download
import shutil
shutil.make_archive('training_results', 'zip', '.', 'results')
print(f"\nDownload: training_results.zip")

try:
    from google.colab import files
    files.download('training_results.zip')
except ImportError:
    print("Not in Colab - manually download training_results.zip")

## Summary

Trained 4 models on EuroSAT:
- Simple CNN (baseline, ~50 epochs)
- ResNet-50 + SE Attention (pretrained, ~30 epochs)
- ViT-Small (pretrained, ~30 epochs)
- ResNet-50 + SSAM custom attention (pretrained, ~30 epochs)

**Saved:** Model weights (.pth), training histories (.json), learning curves

**Next:** Download `training_results.zip`, extract to `Phase2_DL/experiments/results/`, then run `03_Evaluation_Metrics.ipynb` locally.